# Register controls

A controlled operation applies a child when all declared activation values match. Coherent controls retain relative phases across active and inactive branches. This native guide maps all 19 cells of the locked control tutorial.

Adapted from the 2023 Google LLC Apache-2.0 tutorial at Qualtran v0.7.0 revision bcb17bb25360ed68ce3bc084645db3e929799389. Domain ownership and source boundaries are recorded in REGISTER_CONTROL_TUTORIAL_AUDIT.md.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

import torch
from graphviz import Source
from IPython.display import HTML, display

from examples.algorithms.adjoint_programs import serial_atoms, tutorial_definitions
from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.controlled_programs import (
    controlled_each_x_program,
    controlled_serial_program,
    controlled_x_program,
)
from flagquantum.compiler.registers.component_tensor_expansion import (
    expand_component_root_for_tensor,
)
from flagquantum.compiler.registers.control.control_predicate import ControlRegister
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir.program import ProgramRecord
from flagquantum.core.ir.program.register_types import (
    quantum_bit_type,
    signed_register_type,
    unsigned_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.drawer.component_graph import export_component_graph_dot
from flagquantum.simulation.classical_control_activation import (
    classical_controls_active,
)
from flagquantum.simulation.classical_control_patterns import normalize_control_patterns


def show_program(record: ProgramRecord, entry: str = "main") -> Source:
    graph = Source(export_component_graph_dot(record, entry))
    cast(Callable[[object], None], display)(graph)
    return graph


bit = quantum_bit_type()
positive_controls = (Register("ctrl", bit),)
positive_patterns = (ControlRegister(1, (1,)),)
x_definition = next(f for f in tutorial_definitions() if f.name == "x")
x, _ = expand_component_root_for_tensor(
    ProgramRecord((x_definition,)), lambda _: True, entry="x", full_flatten=False
)
x_graph = show_program(x, "tensor_root")
positive = controlled_x_program(positive_controls, positive_patterns)
positive_graph = show_program(positive)


## Interface

Core registers describe the control port types and shapes. Compiler ControlRegister values specify row-major, MSB-first equality clauses. The shared predicate transformation places controls before child data and releases its clean work after measurement-assisted erasure. Simulation evaluates bound classical activation values independently.

A four-bit unsigned word can activate on the integer 6.


In [ ]:
word_controls = (Register("ctrl", unsigned_register_type(4)),)
word_patterns = (ControlRegister(4, (6,)),)
assert classical_controls_active((6,), normalize_control_patterns(6))
word = controlled_x_program(word_controls, word_patterns)
word_graph = show_program(word)


A shaped bit register can activate on the complete pattern [0, 1, 1, 0].


In [ ]:
array_controls = (Register("ctrl", bit, (4,)),)
array_patterns = (ControlRegister(1, (0, 1, 1, 0)),)
assert classical_controls_active(
    ([0, 1, 1, 0],), normalize_control_patterns([0, 1, 1, 0])
)
array = controlled_x_program(array_controls, array_patterns)
array_graph = show_program(array)


Multiple control registers retain their independent array shapes. All clauses must match.


In [ ]:
group_controls = (Register("ctrl1", bit, (2,)), Register("ctrl2", bit, (2,)))
group_patterns = (ControlRegister(1, (0, 1)), ControlRegister(1, (1, 0)))
assert classical_controls_active(
    ([0, 1], [1, 0]), normalize_control_patterns([[0, 1], [1, 0]])
)
grouped = controlled_x_program(group_controls, group_patterns)
grouped_graph = show_program(grouped)


A composite receives control by propagating one positive bit through its stored calls. The three serial atoms preserve their individual identities and call order.


In [ ]:
serial = serial_atoms()
cserial = controlled_serial_program()
serial_graph = show_program(serial)
cserial_graph = show_program(cserial)


## Specialized children

A component author supplies an explicit control adder for a specialized child. The shared builder consumes its control and data inputs and returns their successors. X uses an exact CX leaf.


In [ ]:
positive_bound = bind_register_program(positive, ((0,), (1,)))
positive_counts = dict(Counter(g.name for g in positive_bound.circuit.instructions))
assert positive_counts == {"cx": 1}
print(positive_counts)


An eight-bit equality control has a larger signature. The original cell uses signed QInt(8) with 255 while describing an unsigned value. Its diagram construction succeeds, but 255 is outside the signed execution range. The same metadata is displayed below and its execution refusal is recorded explicitly.


In [ ]:
from flagquantum.drawer.control_symbols import control_register_wire_symbol
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import SymbolClass, register_score_data
from flagquantum.drawer.register_score_html import export_register_score_html

signed_control = Register("ctrl", signed_register_type(8))
signed_metadata = control_register_wire_symbol(signed_control, (), (255,))
assert signed_metadata == ("TextBox", {"text": "255"})
signed_ports = (signed_control, Register("q", bit))
signed_step = RegisterStep(
    "controlled metadata",
    ("ctrl", "q"),
    ("ctrl", "q"),
    input_registers=signed_ports,
    output_registers=signed_ports,
)
signed_score = layout_register_score({"ctrl": 8, "q": 1}, ("ctrl", "q"), (signed_step,))


def signed_symbols(
    step: RegisterStep, register: Register | None, index: tuple[int, ...] | None
) -> tuple[SymbolClass, dict[str, str | int | bool]]:
    if register is None:
        return "Text", {"text": "", "fontsize": 10}
    if register.name == "ctrl":
        assert index is not None
        return control_register_wire_symbol(register, index, (255,))
    return "ModPlus", {}


signed_data = register_score_data(signed_score, register_wire_symbol=signed_symbols)
cast(Callable[[object], None], display)(
    cast(Callable[[str], object], HTML)(export_register_score_html(signed_data))
)
try:
    ControlRegister(8, (255,), "twos_complement").bits()
except ValueError as error:
    signed_refusal = str(error)
else:
    raise AssertionError("signed 255 must fail physical encoding")


## Default fallback

The native hierarchy transformation derives stored composite controls and uses explicit exact leaves for atoms. Core FunctionRecord names retain native identities. The source TestAtom fixture has a synthetic static cost that is independent of its X action.


In [ ]:
catom = controlled_x_program(positive_controls, positive_patterns, atom_name="atom")
catom_graph = show_program(catom)


The shared control implementation has several owners.

- Core retains typed control and child ports.
- Compiler propagates controls, selects specialized adders and owns predicate work lifetimes.
- Simulation checks classical activation and constructs phaseful controlled tensors.
- Resource queries transform declared calls independently of emitted primitives.
- Drawer delegates child titles and draws bit polarities or integer activation values.
- Ecosystem translates admitted operations at the SDK boundary.
- Exact adjoints reverse the existing component graph through its declared inverse leaves.

The locked source implementation now has a direct controlled tensor supplier, despite the older tutorial wording that mentions decomposition only. Its Cirq conversion also directly controls the exported child. The native numerical checks exercise those actual source protocols.


In [ ]:
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

each = controlled_each_x_program()
each_graph = show_program(each)
each_bound = bind_register_program(each, ((0,), (1,), (2,), (3,), (4,)), clean=(5,))
each_counts = dict(Counter(g.name for g in each_bound.circuit.instructions))
assert each_counts["cx"] >= 3 and each_counts["measure"] == 1

initial = torch.zeros(64, dtype=torch.complex128)
initial[48] = 1
circuit = DynamicCircuit(6, inputs=initial, dtype=torch.complex128)
circuit.append(FieldOperation(6, each_bound.circuit.instructions))
result = run_dynamic(circuit, shots=2, seed=29, strategy="trajectory")
expected = torch.zeros_like(initial)
expected[62] = 1
assert torch.allclose(result.final_states.reshape(2, -1), expected.expand(2, -1))


Controlled identity branches require matching input and output spaces. Generic stored-body control requires THRU ports. Existing specialized callbacks may own explicitly verified directional behavior. Predicate workspace starts clean and returns to zero on every applicable measurement branch.
